# NL2SPARQL Test Notebook

## load libraries

In [2]:
from dotenv import load_dotenv
import os
import requests
import spacy
import time
from pathlib import Path
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    PromptTemplate
)
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib.plugins.sparql.parser import parseQuery
from rdflib.plugins.sparql.algebra import translateQuery
import json
import re

# Load environment variables from .env
load_dotenv()

True

## Prompt Construction

In [3]:
def prompt_construction(mentions, linked_entities, question):
    # Read schema
    schema_path = Path("dblp_schema.rdf")
    with schema_path.open("r", encoding="utf-8") as f:
        dblp_schema = f.read()

    # load namespace cheat sheet
    with open("dblp_rdf_schema_cheatsheet.txt", "r", encoding="utf-8") as f:
        namespace_cheatsheet = f.read()

    examples = [
        {
            "question": "Return all papers published by Ian Goodfellow",
            "sparql": """PREFIX dblp: <https://dblp.org/rdf/schema#>
SELECT ?publication ?title WHERE {{
  VALUES ?author {{ <https://dblp.org/pid/43/7940> }}
  ?publication dblp:authoredBy ?author .
  ?publication dblp:title ?title .
}}"""
        },
        {
            "question": "Top 10 most frequent authors who have published at the International Semantic Web Conference (ISWC).",
            "sparql": """PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?name ?affiliation (COUNT(DISTINCT ?publ) as ?freq) (?pers as ?dblp) (SAMPLE(?orcids) as ?orcid)
WHERE {{
  ?stream <https://dblp.org/rdf/schema#primaryStreamTitle> "International Semantic Web Conference" .
  ?publ dblp:publishedInStream ?stream .
  ?publ dblp:authoredBy ?pers .
  ?pers rdfs:label ?name .
  OPTIONAL {{ ?pers dblp:primaryAffiliation ?affiliation . }}
  OPTIONAL {{ ?pers dblp:orcid ?orcids . }}
}}
GROUP BY ?name ?affiliation ?pers
ORDER BY DESC(?freq)
LIMIT 10"""
        },
        {
            "question": "Who are the highly cited coauthors of Andrej Karpathy?",
            "sparql": """PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX cito: <http://purl.org/spar/cito/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?name ?affiliation (COUNT(DISTINCT ?cite) AS ?cites) (?coauthor AS ?dblp) (SAMPLE(?orcids) AS ?orcid) WHERE {{
  VALUES ?author {{ <https://dblp.org/pid/04/9925> }} .
  ?copubl dblp:authoredBy ?author .
  ?copubl dblp:authoredBy ?coauthor .
  FILTER (?author != ?coauthor) .
  ?coauthor rdfs:label ?name .
  OPTIONAL {{ ?coauthor dblp:orcid ?orcids . }}
  OPTIONAL {{ ?coauthor dblp:primaryAffiliation ?affiliation . }}
  ?publ dblp:authoredBy ?coauthor .
  ?publ dblp:omid ?omid .
  ?cite cito:hasCitedEntity ?omid .
}}
GROUP BY ?name ?affiliation ?coauthor
ORDER BY DESC(?cites)
LIMIT 10"""
        }
    ]

    examples_text = "\n\n".join(
        f"Q: {ex['question']}\nSPARQL:\n{ex['sparql']}"
        for ex in examples
    )

    system_prompt_text = f"""
You are a SPARQL query generator for the dblp computer science bibliography RDF dataset.

Schema:
{dblp_schema}

Namespace Cheat Sheet:
{namespace_cheatsheet}

Guidelines:
- Make use of the prefixes provided in the schema (and explained in the namespace cheat sheet)
- Use the extracted and linked entities to inform your query construction
- Return **only** the SPARQL query, nothing else!
- Do **not** add any comments in the query

Examples:
{examples_text}
"""

    # System prompt (static)
    system_prompt = SystemMessagePromptTemplate(
        prompt=PromptTemplate(
            input_variables=[],
            template=system_prompt_text
        )
    )

    # Human prompt (dynamic)
    human_prompt = HumanMessagePromptTemplate(
        prompt=PromptTemplate(
            input_variables=["input_question", "extracted_entities", "linked_entities"],
            template="Question: {input_question}\nExtracted Entities: {extracted_entities}\nLinked Entities: {linked_entities}\n\nSPARQL query:"
        )
    )

    prompt_template = ChatPromptTemplate.from_messages([system_prompt, human_prompt])

    rendered_prompt = prompt_template.format(
        input_question=question,
        extracted_entities=mentions,
        linked_entities=linked_entities,
    )

    return rendered_prompt


## Query Generation via LLM Call

In [4]:
def query_llm(prompt: str) -> str:
    """
    Sends a prompt to the LLM and returns the response text.
    Prints token usage (prompt + completion).
    """

    # --- Read environment variables ---
    OLLAMA_URL = os.getenv("OLLAMA_URL")
    API_KEY = os.getenv("API_KEY")

    if not API_KEY:
        raise ValueError("API_KEY not found. Please set it in your .env file.")
    if not OLLAMA_URL:
        raise ValueError("OLLAMA_URL not found. Please set it in your .env file.")

    # --- Prepare the payload ---
    payload = {
        "model": "gpt-oss:20b",
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    # --- Send the request (streaming) ---
    response = requests.post(OLLAMA_URL, headers=headers, json=payload, stream=True)

    # --- Parse the streamed response ---
    for line in response.iter_lines():
        if not line:
            continue

        data = json.loads(line.decode("utf-8"))
        answer = data["choices"][0]["message"]["content"].strip()

        usage = data.get("usage", {})
        prompt_tokens = usage.get("prompt_tokens")
        completion_tokens = usage.get("completion_tokens")

        print("\nNumber of input (prompt) tokens:", prompt_tokens)
        print("Number of output (completion) tokens:", completion_tokens)

        return answer

    return ""  # Edge case: no output received

## Query Validation

In [5]:
def validate_syntax(query: str):
    try:
        parseQuery(query)
        return True, "✅ SPARQL syntax is valid"
    except Exception as e:
        return False, f"❌ Syntax error: {e}"

## Run Query

In [6]:
def run_query(endpoint, query):
    s = SPARQLWrapper(endpoint)
    s.setMethod("POST")
    s.setTimeout(60)
    s.setReturnFormat(JSON)
    s.addParameter("format", "json")
    s.addCustomHttpHeader("User-Agent", "NL2SPARQL/0.1 (julius.kaltwasser@rwth-aachen.de)")
    s.setQuery(query)
    return s.query().convert()["results"]["bindings"]

## User Question

In [7]:
question = "Who are the highly cited coauthors of Hannah Bast?"
question2 = "Who are the authors of the paper named: Attentions is all you need?"
question3 = "Return the 10 most cited papers of Nicholas Carlini"
question4 = "Database papers published in the Semantic Web Journal."
question5  = "Which papers did Ian Goodfellow publish after 2018?"
question6  = "What are the most cited papers on federated learning?"
question7  = "List all coauthors of Yoshua Bengio."
question8  = "How many papers has Andrew Ng published in NeurIPS?"
question9  = "Show papers that cite “Attention Is All You Need.”"
question10 = "What are the top 5 most cited papers in computer vision?"
question11 = "Who collaborated with Geoffrey Hinton on deep learning papers?"
question12 = "Which authors have published in both CVPR and ICCV?"
question13 = "List all papers with 'graph neural networks' in the title."
question14 = "Which conferences did the paper 'BERT: Pre-training of Deep Bidirectional Transformers' appear in?"
question15 = "Who are the most cited researchers in natural language processing?"
question16 = "Return papers coauthored by Jürgen Schmidhuber and Sepp Hochreiter."
question17 = "Find the average citation count of papers published in ICML 2020."
question18 = "Which journals have published papers about 'adversarial examples'?"
question19 = "Show all papers published by OpenAI authors."
question20 = "Which paper by Yann LeCun has the highest number of citations?"
question21 = "Who are the editors of the journal 'Machine Learning'?"
question22 = "List all papers that reference 'Generative Adversarial Networks.'"
question23 = "Which institutions are most active in quantum computing research?"
question24 = "Find all papers authored by people affiliated with Stanford University in 2023."


## Test

In [17]:
# Function to generate query with prompt construction, validate, run and compare
from typing import Dict, Any
import json

def compare_query(item: Dict[str, Any]):
    """
    Given a dictionary with a userquestion, entities, mentions, and query, this function
    will construct a SPARQL query via the LLM using the prompt_construction function,
    validate the syntax, run both the generated and the original queries, and compare the first 10 results.
    """
    # Extract values
    question = item.get('userquestion') or item.get('question')
    linked_entities = item.get('entities', [])
    # Extract URIs or pass through string list if provided
    uris: list = []
    for ent in linked_entities:
        if isinstance(ent, dict):
            uri = ent.get('uri') or ent.get('label') or ent
        else:
            uri = ent
        uris.append(uri)
    mentions = item.get('mentions')
    if mentions is None:
        mentions = []
        for ent in linked_entities:
            if isinstance(ent, dict):
                label = ent.get('label')
                if label:
                    mentions.append(label)
        if not mentions and question:
            mentions = [question]
    # Construct the prompt and generate a SPARQL query using the LLM
    prompt = prompt_construction(mentions, uris, question)
    try:
        generated_query = query_llm(prompt)
    except Exception as e:
        print(f'Error during query generation: {e}')
        generated_query = ''
    # Validate the syntax of both queries
    is_valid_gen, msg_gen = validate_syntax(generated_query)
    print(f'Generated query validation: {msg_gen}')
    is_valid_gt, msg_gt = validate_syntax(item.get('query', ''))
    print(f'Original query validation: {msg_gt}')
    # Run queries on the DBLP endpoint
    endpoint = 'https://sparql.dblp.org/sparql'
    results_gen: list = []
    results_gt: list = []
    if is_valid_gen and generated_query:
        try:
            results_gen = run_query(endpoint, generated_query)
        except Exception as e:
            print(f'Error running generated query: {e}')
    if is_valid_gt:
        try:
            results_gt = run_query(endpoint, item.get('query', ''))
        except Exception as e:
            print(f'Error running original query: {e}')
    # Compare the first 10 results
    print('Generated query results (first 10):', results_gen[:10])
    print('Original query results (first 10):', results_gt[:10])
    return results_gen, results_gt

def compare_results(generated_results, original_results):
    """
    Compare two lists of result dictionaries and return the count of identical elements.

    Parameters:
        generated_results (list): List of dictionaries from the generated query.
        original_results (list): List of dictionaries from the original query.

    Returns:
        int: Number of matching result dictionaries in both lists.
    """
    # Convert each result dict to a JSON string with sorted keys
    gen_set = set(json.dumps(item, sort_keys=True) for item in generated_results)
    orig_set = set(json.dumps(item, sort_keys=True) for item in original_results)

    # Count how many items are in both sets
    return len(gen_set & orig_set)

In [19]:
# Run compare_query for only the first 3 items
import json

dataset_path = 'askDBLP_dataset.json'
with open(dataset_path, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

# Restrict to first 3 items
subset = dataset[:3]
total_items = len(subset)

perfect_matches = 0
scores = []

for idx, item in enumerate(subset):
    question = item.get('question')
    entities = item.get('entities', [])
    query = item.get('query', '')

    # Collect mentions
    mentions = []
    for ent in entities:
        if isinstance(ent, dict):
            label = ent.get('label')
            if label:
                mentions.append(label)

    compare_input = {
        'userquestion': question,
        'entities': entities,
        'mentions': mentions,
        'query': query
    }

    print(f"\nProcessing item {idx+1}/{total_items}: {question}")

    generated_results, original_results = compare_query(compare_input)

    score = compare_results(generated_results, original_results)
    scores.append(score)

    print(f"Matching elements: {score}/10")

    if score == 10:
        perfect_matches += 1

# Final summary
print("\n=======================")
print("      FINAL RESULT     ")
print("=======================\n")
print(f"Items with perfect match (10/10): {perfect_matches} out of {total_items}")
print(f"All scores per item: {scores}")



Processing item 1/3: Who are the highly cited coauthors of Hannah Bast?

Number of input (prompt) tokens: 8192
Number of output (completion) tokens: 1362
Generated query validation: ✅ SPARQL syntax is valid
Original query validation: ✅ SPARQL syntax is valid
Generated query results (first 10): [{'name': {'type': 'literal', 'value': 'Thomas Brox'}, 'affiliation': {'type': 'literal', 'value': 'University of Freiburg, Department of Computer Science, Germany'}, 'cites': {'datatype': 'http://www.w3.org/2001/XMLSchema#int', 'type': 'literal', 'value': '61353'}, 'dblp': {'type': 'uri', 'value': 'https://dblp.org/pid/97/4586'}, 'orcid': {'type': 'uri', 'value': 'https://orcid.org/0000-0002-6282-8861'}}, {'name': {'type': 'literal', 'value': 'Gerhard Weikum'}, 'affiliation': {'type': 'literal', 'value': 'Max Planck Institute for Informatics, Saarbrücken, Germany'}, 'cites': {'datatype': 'http://www.w3.org/2001/XMLSchema#int', 'type': 'literal', 'value': '14292'}, 'dblp': {'type': 'uri', 'value